In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("SparkSkewPerformanceLab")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# print("Spark version:", spark.version)
# print("Default parallelism:", spark.sparkContext.defaultParallelism)


D:\Personal\Study\Learning\Data Engineer\De-learning-journey\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
N = 2_000_000

df = (
    spark.range(N)
    .withColumn(
        "customer_id",
        F.when(F.col("id") < 1_800_000, F.lit(1))
         .otherwise((F.col("id") % 9999) + 2)
    )
    .withColumn(
        "amount",
        (F.rand(seed=42) * 1000).cast("double")
    )
    .withColumn(
        "category",
        F.when(F.col("id") % 5 == 0, "A")
         .when(F.col("id") % 5 == 1, "B")
         .when(F.col("id") % 5 == 2, "C")
         .when(F.col("id") % 5 == 3, "D")
         .otherwise("E")
    )
    .select("id", "customer_id", "amount", "category")
)

# print("Partitions:", df.rdd.getNumPartitions())
# print("Rows:", df.count())

In [3]:
de = df.groupBy("customer_id") \
  .count() \
  .orderBy(F.desc("count"))
# de.show(10)

In [4]:
partition_sizes = (
    df.rdd
      .mapPartitions(lambda rows: [sum(1 for _ in rows)])
      .collect()
)

# print(partition_sizes)

In [5]:
result = (
    df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.max("amount").alias("max_amount"),
        F.min("amount").alias("min_amount")
    )
)

# result.show(10)

In [6]:
# result.explain("formatted")

In [7]:
result_partition_sizes = (
    result.rdd
          .mapPartitions(lambda rows: [sum(1 for _ in rows)])
          .collect()
)

# print(result_partition_sizes)

In [8]:
customers = (
    spark.range(2, 10002)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer_"), F.col("customer_id"))
    )
    .withColumn(
        "region",
        F.when(F.col("customer_id") % 4 == 0, "South")
         .when(F.col("customer_id") % 4 == 1, "North")
         .when(F.col("customer_id") % 4 == 2, "East")
         .otherwise("West")
    )
)

# customers.show(10)

In [9]:
joined = (
    df.join(
        customers,
        on="customer_id",
        how="inner"
    )
)

In [10]:
# joined.show(10)

In [11]:
regional_sales = (
    joined
    .groupBy("region", "category")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("amount").alias("revenue"),
        F.avg("amount").alias("avg_transaction"),
        F.max("amount").alias("largest_transaction")
    )
    .orderBy(F.desc("revenue"))
)

# regional_sales.show()

In [12]:
window_spec = (
    Window
    .partitionBy("region")
    .orderBy(F.desc("revenue"))
)

ranked = (
    regional_sales
    .withColumn(
        "region_rank",
        F.row_number().over(window_spec)
    )
)

ranked.show()

+------+--------+------------+-----------------+------------------+-------------------+-----------+
|region|category|transactions|          revenue|   avg_transaction|largest_transaction|region_rank|
+------+--------+------------+-----------------+------------------+-------------------+-----------+
| North|       C|        9997|5035667.856164149| 503.7179009867109|  999.9375951796874|          1|
| North|       B|        9997|5022642.365806164| 502.4149610689371|  999.9143521980067|          2|
| North|       A|        9997|5021866.764549508|502.33737766825124|  999.7905457323111|          3|
| North|       E|        9997|5012717.882955273|501.42221496001525|  999.9560599109791|          4|
| North|       D|        9997|4976072.870557661| 497.7566140399781|  999.9237781522417|          5|
|  West|       A|       10001|5013836.736265204|501.33354027249317|  999.9469424328635|          1|
|  West|       C|       10001|4992559.706298178|499.20605002481534|  999.8971043432771|          2|


In [13]:
# ranked.collect()

In [15]:
spark.stop()